# Embedding Quality — Quick Wins

t-SNE plots comparing `model_no_orthogonal` vs `model_orthogonal`, colored by **team** and **year**.

In [ ]:
import sys, os, warnings
sys.path.append(os.path.abspath("../src"))

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE

from models.pipeline_fusion import F1OrthogonalPipeline
from relbench.datasets import get_dataset
from train import filter_db_by_years, build_graph, _build_instances
import config as cfg

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
%matplotlib inline

In [ ]:
# ---------------------------------------------------------------------------
# Config — adjust these if your setup differs
# ---------------------------------------------------------------------------
LATENT_DIM = 32          # must match training latent_dim
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TOP_N_TEAMS = 8          # how many teams to color individually (rest → gray)
PERPLEXITY = 30          # t-SNE perplexity

MODEL_NAMES = ["model_no_orthogonal", "model_orthogonal"]
MODEL_LABELS = ["No Orthogonal", "Orthogonal"]

for name in MODEL_NAMES:
    path = f"../output/models/{name}.pth"
    if not os.path.exists(path):
        raise FileNotFoundError(f"Model not found: {path}. Train it first on the GPU machine.")

In [ ]:
# ---------------------------------------------------------------------------
# 1. Load DB and build metadata (driver → team, driver → year)
# ---------------------------------------------------------------------------
print("Loading DB & metadata...")

dataset = get_dataset(cfg.RELBENCH_DATASET, download=True)
db = dataset.get_db(upto_test_timestamp=False)
db = filter_db_by_years(db, 2000, 2023)

instances = _build_instances(db)

# Per-driver: most frequent constructor → team name
driver_team_id = instances.groupby("driverId")["constructorId"] \
    .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else x.iloc[0])
driver_year_mean = instances.groupby("driverId")["year"].mean()

# Human-readable names
cons_id_to_name = db.table_dict["constructors"].df \
    .set_index("constructorId")["name"].to_dict()
driver_id_to_name = db.table_dict["drivers"].df \
    .set_index("driverId").apply(
        lambda r: f"{r['forename']} {r['surname']}", axis=1
    ).to_dict()

n_drivers = len(driver_id_to_name)
n_constructors = len(cons_id_to_name)
print(f"Drivers: {n_drivers}, Constructors: {n_constructors}, Instances: {len(instances)}")

In [ ]:
# ---------------------------------------------------------------------------
# 2. Build graph
# ---------------------------------------------------------------------------
print("Building graph...")

graph_data, node_to_col_names_dict, node_to_col_stats = build_graph(db)

num_nodes_dict = {nt: graph_data[nt].num_nodes for nt in graph_data.node_types}

# Keep CPU copy for tf_dict (TensorFrame doesn't move with .to())
graph_cpu = graph_data
graph_data = graph_data.to(DEVICE)

print("Node types:", graph_data.node_types)
print("Edge types:", len(graph_data.edge_types))

In [ ]:
# ---------------------------------------------------------------------------
# 3. Load models and extract ALL driver + constructor embeddings
# ---------------------------------------------------------------------------
def load_model(name):
    model = F1OrthogonalPipeline(
        num_nodes_dict=num_nodes_dict,
        node_to_col_names_dict=node_to_col_names_dict,
        node_to_col_stats=node_to_col_stats,
        latent_dim=LATENT_DIM,
    ).to(DEVICE)
    path = f"../output/models/{name}.pth"
    model.load_state_dict(torch.load(path, map_location=DEVICE))
    model.eval()
    return model

embeddings = {}  # label -> (drivers_np, constructors_np)

for name, label in zip(MODEL_NAMES, MODEL_LABELS):
    print(f"Loading {label}...")
    model = load_model(name)

    with torch.no_grad():
        # Encoder processes CPU tf_dict → outputs tensors on model device
        x_dict = model.encoder(graph_cpu.tf_dict) if model.encoder is not None else graph_cpu.x_dict
        out_dict = model.graph_encoder(x_dict, graph_data.edge_index_dict)

    drv = out_dict["drivers"].cpu().numpy()
    cons = out_dict["constructors"].cpu().numpy()
    embeddings[label] = (drv, cons)
    print(f"  drivers: {drv.shape}, constructors: {cons.shape}")

In [ ]:
# ---------------------------------------------------------------------------
# 4. Joint t-SNE (both models in the same 2D space for direct comparison)
# ---------------------------------------------------------------------------
print("Running t-SNE...")

# Concatenate driver embeddings from both models
all_drv = np.concatenate([embeddings[l][0] for l in MODEL_LABELS], axis=0)

perp = min(PERPLEXITY, n_drivers - 1)
tsne = TSNE(n_components=2, perplexity=perp, random_state=42, verbose=1)
all_2d = tsne.fit_transform(all_drv)

# Split back
drv_2d = {}
offset = 0
for label in MODEL_LABELS:
    n = embeddings[label][0].shape[0]
    drv_2d[label] = all_2d[offset:offset + n]
    offset += n

In [ ]:
# ---------------------------------------------------------------------------
# 5. Build unified plot DataFrame
# ---------------------------------------------------------------------------
rows = []
for label in MODEL_LABELS:
    xy = drv_2d[label]
    for driver_id in range(xy.shape[0]):
        rows.append({
            "x": xy[driver_id, 0],
            "y": xy[driver_id, 1],
            "model": label,
            "driver": driver_id_to_name.get(driver_id, f"id_{driver_id}"),
            "team": cons_id_to_name.get(driver_team_id.get(driver_id, -1), "Unknown"),
            "year": driver_year_mean.get(driver_id, np.nan),
        })

df = pd.DataFrame(rows)

# Top-N teams for coloring (rest grouped as "Other")
top_teams = df["team"].value_counts().head(TOP_N_TEAMS).index.tolist()
palette = sns.color_palette("tab10", n_colors=len(top_teams))
team_color_map = dict(zip(top_teams, palette))

df["team_color"] = df["team"].apply(
    lambda t: t if t in top_teams else "Other"
)
print(f"Top {TOP_N_TEAMS} teams: {top_teams}")

In [ ]:
# ---------------------------------------------------------------------------
# 6a. Side-by-side: colored by TEAM
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, label in zip(axes, MODEL_LABELS):
    subset = df[df["model"] == label]

    # "Other" first (background layer)
    other = subset[subset["team_color"] == "Other"]
    ax.scatter(other["x"], other["y"], s=8, alpha=0.25, c="lightgray")

    # Then top teams
    for team in top_teams:
        td = subset[subset["team"] == team]
        ax.scatter(td["x"], td["y"], s=22, alpha=0.75,
                   color=team_color_map[team], label=team, edgecolors="none")

    ax.set_title(label, fontsize=14, weight="bold")
    ax.set_xlabel("t-SNE 1")
    ax.set_ylabel("t-SNE 2")

handles, labels_ = axes[0].get_legend_handles_labels()
fig.legend(handles, labels_, loc="lower center", ncol=len(top_teams),
           fontsize=8, frameon=True, bbox_to_anchor=(0.5, -0.06))
fig.suptitle("Driver Embeddings — colored by Team", fontsize=16, y=0.98)
plt.tight_layout(rect=[0, 0.08, 1, 1])
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# 6b. Side-by-side: colored by YEAR
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

vmin, vmax = df["year"].min(), df["year"].max()

for ax, label in zip(axes, MODEL_LABELS):
    subset = df[df["model"] == label]
    sc = ax.scatter(subset["x"], subset["y"], s=18, alpha=0.7,
                    c=subset["year"], cmap="viridis", vmin=vmin, vmax=vmax)
    ax.set_title(label, fontsize=14, weight="bold")
    ax.set_xlabel("t-SNE 1")
    ax.set_ylabel("t-SNE 2")

cbar = fig.colorbar(sc, ax=axes, fraction=0.025, pad=0.02)
cbar.set_label("Mean Year", fontsize=11)
fig.suptitle("Driver Embeddings — colored by Year", fontsize=16, y=0.98)
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# 7 (bonus). Pairwise cosine histogram — driver vs constructor
# ---------------------------------------------------------------------------
from sklearn.preprocessing import normalize

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax, label in zip(axes, MODEL_LABELS):
    drv, cons = embeddings[label]
    drv_n = normalize(drv, norm="l2")
    cons_n = normalize(cons, norm="l2")
    cos_mat = np.dot(drv_n, cons_n.T)  # (N_drv, N_cons)

    all_cos = cos_mat.ravel()

    ax.hist(all_cos, bins=80, alpha=0.7, color="steelblue", edgecolor="white", linewidth=0.3)
    ax.axvline(x=0, color="red", linestyle="--", linewidth=1.2, alpha=0.7)
    ax.set_title(f"{label}\nμ={all_cos.mean():.4f}  σ={all_cos.std():.4f}",
                 fontsize=13, weight="bold")
    ax.set_xlabel("Cosine similarity (driver × constructor)")
    if ax == axes[0]:
        ax.set_ylabel("Count")

fig.suptitle("Pairwise Cosine Similarity: Driver ↔ Constructor", fontsize=15)
plt.tight_layout()
plt.show()